# Log–Log Rate–Time b-Factor by API

Upload a CSV containing three columns:

- **API** — well identifier
- **Date** — formatted as `MM/DD/YYYY`
- **Rate** — oil rate

For every unique API, the notebook:

1. Sorts observations by date.
2. Defines elapsed time as `date - first valid date + 1 day` so the first point is day 1 and can be used on a logarithmic axis.
3. Fits a straight line in log space:

\[
\log_{10}(q) = m\log_{10}(t) + c
\]

4. Calculates the RTA b-factor estimate:

\[
b = -\frac{1}{m}
\]

Only positive, finite rates are used. APIs with fewer than the required number of usable points are retained in the output with an explanatory status.

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except ImportError:
    from IPython.display import display
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (9, 6)
plt.rcParams["figure.dpi"] = 130

## 1. Load the CSV

Use the upload button below, or set `CSV_FILENAME` in the fallback cell to a CSV already present in the JupyterLite file browser.

In [ ]:
uploaded_df = None

if WIDGETS_AVAILABLE:
    uploader = widgets.FileUpload(accept=".csv", multiple=False, description="Upload CSV")
    output = widgets.Output()

    def _read_upload(change):
        global uploaded_df
        output.clear_output()
        with output:
            try:
                value = uploader.value
                if not value:
                    print("Choose a CSV file.")
                    return

                if isinstance(value, dict):
                    item = next(iter(value.values()))
                else:
                    item = value[0]

                content = item["content"]
                uploaded_df = pd.read_csv(io.BytesIO(bytes(content)), dtype=str)
                print(f"Loaded {len(uploaded_df):,} rows.")
                display(uploaded_df.head())
                print("Columns:", list(uploaded_df.columns))
            except Exception as exc:
                print(f"Could not read CSV: {exc}")

    uploader.observe(_read_upload, names="value")
    display(uploader, output)
else:
    print("ipywidgets is unavailable. Use the filename fallback cell below.")

In [ ]:
# Fallback: upload the CSV through JupyterLite's file browser,
# enter its filename here, and run this cell.
CSV_FILENAME = ""  # Example: "oil_rates.csv"

if CSV_FILENAME:
    uploaded_df = pd.read_csv(CSV_FILENAME, dtype=str)
    print(f"Loaded {len(uploaded_df):,} rows.")
    display(uploaded_df.head())
    print("Columns:", list(uploaded_df.columns))

## 2. Configure the columns and fitting controls

Column names must exactly match the CSV headers. The optional fitting bounds are elapsed days measured separately from each API's first valid date.

In [ ]:
API_COLUMN = "API"
DATE_COLUMN = "date"
RATE_COLUMN = "rate"
DATE_FORMAT = "%m/%d/%Y"

# Optional common fitting interval for every API, in elapsed days.
FIT_START_DAY = None   # Example: 30
FIT_END_DAY = None     # Example: 1000

# Quality/control settings.
MIN_FIT_POINTS = 2
PLOT_EACH_API = True
MAX_PLOTS = None       # Example: 25; use None to plot every successful API

# Optional: restrict analysis to specified APIs. Leave as None for all APIs.
API_FILTER = None      # Example: ["42000000000000", "42000000000001"]

## 3. Calculate one b-factor per API

In [ ]:
def calculate_b_factors_by_api(
    data,
    api_column,
    date_column,
    rate_column,
    date_format="%m/%d/%Y",
    fit_start_day=None,
    fit_end_day=None,
    min_fit_points=2,
    plot_each_api=True,
    max_plots=None,
    api_filter=None,
):
    """Calculate log-log slope and b = -1/m independently for each API."""
    if data is None:
        raise ValueError("No CSV is loaded. Upload a file or set CSV_FILENAME first.")

    required = [api_column, date_column, rate_column]
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise KeyError(
            f"Missing column(s): {missing}. Available columns: {list(data.columns)}"
        )

    work = data[required].copy()
    work.columns = ["API", "date", "rate"]

    # Keep API as text so leading zeros and long identifiers are preserved.
    work["API"] = work["API"].astype("string").str.strip()
    work["date"] = pd.to_datetime(
        work["date"].astype("string").str.strip(),
        format=date_format,
        errors="coerce",
    )
    work["rate"] = pd.to_numeric(work["rate"], errors="coerce")
    work = work.replace([np.inf, -np.inf], np.nan)
    work = work.dropna(subset=["API", "date", "rate"])
    work = work[work["API"] != ""]

    if api_filter is not None:
        selected = {str(api).strip() for api in api_filter}
        work = work[work["API"].isin(selected)]

    if work.empty:
        raise ValueError("No valid API/date/rate rows remain after cleaning.")

    result_rows = []
    fitted_frames = []
    plots_created = 0

    for api, group in work.groupby("API", sort=True):
        group = group.sort_values("date").copy()
        total_rows = len(group)
        first_date = group["date"].min()
        last_date = group["date"].max()

        # Add 1 so the first observation is elapsed day 1, not zero.
        group["elapsed_days"] = (group["date"] - first_date).dt.days + 1
        group = group[(group["elapsed_days"] > 0) & (group["rate"] > 0)].copy()
        positive_rows = len(group)

        fit = group.copy()
        if fit_start_day is not None:
            fit = fit[fit["elapsed_days"] >= fit_start_day]
        if fit_end_day is not None:
            fit = fit[fit["elapsed_days"] <= fit_end_day]

        base_result = {
            "API": api,
            "first_date": first_date.date().isoformat(),
            "last_date": last_date.date().isoformat(),
            "input_rows": total_rows,
            "positive_rate_rows": positive_rows,
            "fit_points": len(fit),
            "fit_start_elapsed_day": np.nan if fit.empty else fit["elapsed_days"].min(),
            "fit_end_elapsed_day": np.nan if fit.empty else fit["elapsed_days"].max(),
        }

        if len(fit) < min_fit_points:
            result_rows.append({
                **base_result,
                "slope_m": np.nan,
                "intercept_c": np.nan,
                "b_factor_negative_inverse_slope": np.nan,
                "r_squared_log_space": np.nan,
                "status": f"insufficient fit points (<{min_fit_points})",
            })
            continue

        log_t = np.log10(fit["elapsed_days"].to_numpy(dtype=float))
        log_q = np.log10(fit["rate"].to_numpy(dtype=float))

        # A fit is undefined if all elapsed-day values are identical.
        if np.unique(log_t).size < 2:
            result_rows.append({
                **base_result,
                "slope_m": np.nan,
                "intercept_c": np.nan,
                "b_factor_negative_inverse_slope": np.nan,
                "r_squared_log_space": np.nan,
                "status": "fit days are not distinct",
            })
            continue

        slope, intercept = np.polyfit(log_t, log_q, 1)
        predicted_log_q = slope * log_t + intercept
        ss_res = np.sum((log_q - predicted_log_q) ** 2)
        ss_tot = np.sum((log_q - np.mean(log_q)) ** 2)
        r_squared = np.nan if np.isclose(ss_tot, 0.0) else 1 - ss_res / ss_tot
        b_factor = np.inf if np.isclose(slope, 0.0) else -1.0 / slope

        if slope < 0:
            status = "ok"
        elif np.isclose(slope, 0.0):
            status = "near-zero slope; b is infinite"
        else:
            status = "positive slope; b is negative"

        result_rows.append({
            **base_result,
            "slope_m": slope,
            "intercept_c": intercept,
            "b_factor_negative_inverse_slope": b_factor,
            "r_squared_log_space": r_squared,
            "status": status,
        })

        fit_output = fit.copy()
        fit_output["slope_m"] = slope
        fit_output["b_factor_negative_inverse_slope"] = b_factor
        fit_output["r_squared_log_space"] = r_squared
        fitted_frames.append(fit_output)

        should_plot = plot_each_api and (max_plots is None or plots_created < max_plots)
        if should_plot:
            line_t = np.geomspace(
                fit["elapsed_days"].min(), fit["elapsed_days"].max(), 200
            )
            line_q = 10 ** (intercept + slope * np.log10(line_t))

            fig, ax = plt.subplots()
            ax.loglog(
                group["elapsed_days"], group["rate"], "o",
                markersize=4, alpha=0.6, label="Positive-rate data"
            )
            ax.loglog(
                fit["elapsed_days"], fit["rate"], "o",
                markersize=5, label="Points used in fit"
            )
            ax.loglog(
                line_t, line_q, "-", linewidth=2,
                label=f"Fit: m={slope:.4f}, b={b_factor:.4f}"
            )
            ax.set_xlabel("Elapsed time from first date (days; first point = day 1)")
            ax.set_ylabel("Oil rate")
            ax.set_title(f"API {api}: Log–Log Oil Rate vs Time")
            ax.grid(True, which="both", alpha=0.3)
            ax.legend()
            ax.text(
                0.03, 0.03,
                f"n = {len(fit)}\n"
                f"m = {slope:.6f}\n"
                f"b = -1/m = {b_factor:.6f}\n"
                f"R² = {r_squared:.6f}",
                transform=ax.transAxes,
                va="bottom",
                bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
            )
            plt.show()
            plots_created += 1

    results = pd.DataFrame(result_rows).sort_values("API").reset_index(drop=True)
    fitted_points = (
        pd.concat(fitted_frames, ignore_index=True)
        if fitted_frames
        else pd.DataFrame()
    )

    return results, fitted_points


results_by_api, fitted_points_by_api = calculate_b_factors_by_api(
    uploaded_df,
    api_column=API_COLUMN,
    date_column=DATE_COLUMN,
    rate_column=RATE_COLUMN,
    date_format=DATE_FORMAT,
    fit_start_day=FIT_START_DAY,
    fit_end_day=FIT_END_DAY,
    min_fit_points=MIN_FIT_POINTS,
    plot_each_api=PLOT_EACH_API,
    max_plots=MAX_PLOTS,
    api_filter=API_FILTER,
)

display(results_by_api)
print(f"Processed {len(results_by_api):,} unique APIs.")
print(results_by_api["status"].value_counts(dropna=False))

## 4. Export the results

Run the cell below to create CSV files in the JupyterLite file browser.

In [ ]:
results_by_api.to_csv("rta_b_factors_by_api.csv", index=False)

if not fitted_points_by_api.empty:
    fitted_points_by_api.to_csv("rta_fitted_points_by_api.csv", index=False)

print("Saved rta_b_factors_by_api.csv")
if not fitted_points_by_api.empty:
    print("Saved rta_fitted_points_by_api.csv")

### Output fields

The main output `rta_b_factors_by_api.csv` contains one row per API, including:

- `slope_m`
- `b_factor_negative_inverse_slope`
- `r_squared_log_space`
- number of input, positive-rate, and fitted points
- fitted elapsed-day range
- a `status` field identifying successful, insufficient-data, flat-slope, or positive-slope cases

A positive fitted slope produces a negative calculated b-factor and is explicitly flagged rather than silently discarded.